# Provisional weekly connection behavior
Offline analysis of the saved profiling cache. No API calls. This proposes a descriptive metric, not a drive-failure definition. Week labels use **device wall-calendar dates provisionally** because source offsets are absent. Do not claim a confirmed reporting timezone or a causal release regression.

In [1]:
from pathlib import Path
import json
import pandas as pd

CACHE = Path("data/profiling")
scope = json.loads((CACHE / "scope.json").read_text())
as_of = pd.Timestamp(scope["as_of"])
raw = {}
for name in ["drives", "warnings"]:
    state = json.loads((CACHE / f"{name}.json").read_text())
    if not state["complete"]:
        raise RuntimeError(f"{name}: finish extraction first")
    frame = pd.json_normalize(state["records"])
    receipt = pd.to_datetime(frame["received_at"], utc=True, errors="coerce")
    raw[name] = frame.loc[receipt.notna() & receipt.le(as_of)].copy()
print("Analysis receipt cutoff:", as_of)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


Analysis receipt cutoff: 2026-10-27 15:21:56.773109+00:00


## 1. Verify business identity before collapsing replays
Compare every observed field except `received_at`. Exclude IDs with business-field conflicts instead of choosing a winner. For agreeing records retain the earliest receipt as availability time and latest receipt as replay metadata.

In [2]:
clean = {}
audit = []
for name, key in [("drives", "session_id"), ("warnings", "warning_id")]:
    frame = raw[name]
    business = frame.drop(columns="received_at").drop_duplicates()
    conflicting = business.loc[business[key].notna() & business[key].duplicated(keep=False), key].unique()
    safe = frame.loc[frame[key].notna() & ~frame[key].isin(conflicting)].copy()
    receipts = safe.groupby(key)["received_at"].agg(first_received_at="min", last_received_at="max", report_count="size")
    clean[name] = safe.drop_duplicates(key).drop(columns="received_at").join(receipts, on=key)
    assert clean[name][key].is_unique
    audit.append({"dataset": name, "raw_rows": len(frame), "missing_id_rows": int(frame[key].isna().sum()), "business_conflicting_ids": len(conflicting), "canonical_records": len(clean[name]), "collapsed_replay_rows": len(safe)-len(clean[name])})
audit = pd.DataFrame(audit)
display(audit)
d = clean["drives"].copy()
w = clean["warnings"].copy()

,dataset,raw_rows,missing_id_rows,business_conflicting_ids,canonical_records,collapsed_replay_rows
0,drives,224194,0,0,219848,4346
1,warnings,315601,0,0,315601,0


## 2. Timestamp evidence
Device fields are parsed without attaching a timezone. Device wall time minus UTC receipt clock is an offset/lag diagnostic, **not latency**. Country is insufficient to infer a device timezone. Daylight-saving changes also prevent assuming a constant offset. Warning timestamps cannot reliably establish week boundaries.

In [3]:
for frame, columns in [(d, ["started_at", "ended_at"]), (w, ["device_time"])]:
    for column in columns:
        if frame[column].dropna().str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", regex=True).any():
            raise RuntimeError("Unexpected timezone-bearing device value; review parsing")
        frame[column] = pd.to_datetime(frame[column], format="mixed", errors="coerce")
d["first_receipt_utc"] = pd.to_datetime(d["first_received_at"], utc=True)
d["wall_minus_receipt_hours"] = (d["ended_at"] - d["first_receipt_utc"].dt.tz_localize(None)).dt.total_seconds()/3600
display(d.groupby("country", dropna=False)["wall_minus_receipt_hours"].agg(["count", "median", "min", "max"]).round(2))
print("Warning timestamp range:", w["device_time"].min(), "to", w["device_time"].max())
clock_future = w["device_time"].gt(as_of.tz_localize(None) + pd.Timedelta(days=2))
print("Warnings more than two calendar days beyond cutoff (clock flag):", int(clock_future.sum()))
matched = w.merge(d[["session_id", "user_id", "started_at", "ended_at"]], on="session_id", how="left", suffixes=("_warning", "_drive"), indicator=True, validate="many_to_one")
is_match = matched["_merge"].eq("both")
comparable = is_match & matched["device_time"].notna() & matched["started_at"].notna() & matched["ended_at"].notna()
outside = comparable & (matched["device_time"].lt(matched["started_at"]) | matched["device_time"].gt(matched["ended_at"]))
display(pd.Series({"warning_rows": len(w), "matched_warnings": int(is_match.sum()), "match_pct": round(100*is_match.mean(), 3), "unmatched": int((~is_match).sum()), "outside_drive_interval": int(outside.sum()), "user_id_mismatches": int((is_match & matched["user_id_warning"].notna() & matched["user_id_drive"].notna() & matched["user_id_warning"].ne(matched["user_id_drive"])).sum())}, name="value").to_frame())

Warning timestamp range: 2026-08-15 06:11:49 to 2038-01-19 03:14:07
Warnings more than two calendar days beyond cutoff (clock flag): 150


,count,median,min,max
country,,,,
DE,126581,1.98,-237.60,2.0
DK,32173,1.98,-234.09,2.0
ES,13155,1.98,-233.25,2.0
FR,11555,1.98,-227.02,2.0
GB,23553,0.98,-237.34,1.0
IT,10998,1.98,-235.19,2.0
NaN,1833,1.98,-207.38,2.0


,value
warning_rows,315601.000
matched_warnings,315449.000
match_pct,99.952
unmatched,152.000
outside_drive_interval,150.000
user_id_mismatches,0.000


## 3. Eligibility and provisional Monday–Sunday weeks
Denominator: unique reported sessions with parseable start/end, end at or after start, positive numeric duration agreeing with elapsed time within one second, and integer connection count at least one. Numerator: those eligible sessions with count > 1. No missing/invalid values are converted to zero. Deduplicated report volume is displayed separately. Missing platform/version values stay in an explicit unknown group. Weeks are start-date wall-calendar weeks; current week is excluded. Volume is still subject to late reports and sessions never reported.

In [4]:
d["duration_s"] = pd.to_numeric(d["duration_s"], errors="coerce")
d["connection_count"] = pd.to_numeric(d["connection_count"], errors="coerce")
elapsed = (d["ended_at"]-d["started_at"]).dt.total_seconds()
d["eligible"] = (d["started_at"].notna() & d["ended_at"].notna() & elapsed.ge(0) & d["duration_s"].gt(0) & (d["duration_s"]-elapsed).abs().le(1) & d["connection_count"].ge(1) & d["connection_count"].mod(1).eq(0))
d["multiple_connections"] = d["connection_count"].gt(1)
d["week"] = d["started_at"].dt.to_period("W-SUN").dt.start_time
today = as_of.tz_convert("Europe/Copenhagen").tz_localize(None).normalize()
this_monday = today-pd.Timedelta(days=today.dayofweek)
last_week = this_monday-pd.Timedelta(days=7)
previous_week = this_monday-pd.Timedelta(days=14)
print("Provisional comparison weeks:", previous_week.date(), "and", last_week.date())
display(pd.Series({"canonical_reported_drives": len(d), "eligible_drives": int(d["eligible"].sum()), "excluded_from_metric": int((~d["eligible"]).sum())}, name="rows").to_frame())
cohort = d.loc[d["eligible"] & d["week"].isin([previous_week, last_week])].copy()
for col in ["platform", "app_version", "device"]:
    cohort[col] = cohort[col].fillna("unknown")

def summarize(frame, keys):
    out = frame.groupby(keys, dropna=False).agg(eligible_drives=("session_id", "size"), drivers=("user_id", "nunique"), drives_with_multiple_connections=("multiple_connections", "sum"), median_duration_s=("duration_s", "median"), p95_connections=("connection_count", lambda s:s.quantile(.95)))
    out["multiple_connection_pct"] = 100*out["drives_with_multiple_connections"]/out["eligible_drives"]
    involved = frame.loc[frame["multiple_connections"]].groupby(keys, dropna=False)["user_id"].nunique()
    out["drivers_with_multiple_connection_drives"] = involved.reindex(out.index, fill_value=0)
    return out.round(3)
weekly = summarize(cohort, ["week"])
platforms = summarize(cohort, ["week", "platform"])
releases = summarize(cohort, ["week", "platform", "app_version"])
display(weekly)
display(platforms)
display(releases)
# Separate volume reconciliation: all canonical reports versus metric denominator.
display(d.loc[d["week"].isin([previous_week,last_week])].groupby("week").agg(reported_drives=("session_id","size"), eligible_drives=("eligible","sum")))
assert cohort["session_id"].is_unique
assert weekly["eligible_drives"].sum() == len(cohort)
assert weekly["drives_with_multiple_connections"].le(weekly["eligible_drives"]).all()
assert weekly["multiple_connection_pct"].between(0,100).all()

Provisional comparison weeks: 2026-10-12 and 2026-10-19


,rows
canonical_reported_drives,219848
eligible_drives,219848
excluded_from_metric,0


,eligible_drives,drivers,drives_with_multiple_connections,median_duration_s,p95_connections,multiple_connection_pct,drivers_with_multiple_connection_drives
week,,,,,,,
2026-10-12,21711,2462,1519,1000.0,2.0,6.996,804
2026-10-19,21595,2472,1422,990.0,2.0,6.585,725


eligible_drives  drivers  drives_with_multiple_connections  median_duration_s  p95_connections  multiple_connection_pct  \
week       platform                                                                                                                            
2026-10-12 android             10093     1126                              1287              994.0              3.0                   12.751   
           ios                 11618     1337                               232             1004.0              1.0                    1.997   
2026-10-19 android             10071     1124                              1320              987.0              3.0                   13.107   
           ios                 11524     1349                               102              992.5              1.0                    0.885   

                     drivers_with_multiple_connection_drives  
week       platform                                           
2026-10-12 android                                       650  
           ios                                           154  
2026-10-19 android                                       671  
           ios                                            54

eligible_drives  drivers  drives_with_multiple_connections  median_duration_s  p95_connections  multiple_connection_pct  \
week       platform app_version                                                                                                                            
2026-10-12 android  6.1.0                    271       27                                22              999.0              2.5                    8.118   
                    6.2.0                   9822     1108                              1265              994.0              3.0                   12.879   
           ios      6.1.0                    107        1                                 0              854.0              1.0                    0.000   
                    6.1.1                   2344      380                               232             1016.0              3.0                    9.898   
                    6.3.0                   9167     1158                                 0             1001.0              1.0                    0.000   
2026-10-19 android  6.1.0                    112       10                                 3              972.5              1.0                    2.679   
                    6.2.0                   9959     1119                              1317              987.0              3.0                   13.224   
           ios      6.1.0                    103        1                                 0             1056.0              1.0                    0.000   
                    6.1.1                    772      117                               102              952.0              3.0                   13.212   
                    6.3.0                  10649     1289                                 0              996.0              1.0                    0.000   

                                 drivers_with_multiple_connection_drives  
week       platform app_version                                           
2026-10-12 android  6.1.0                                             15  
                    6.2.0                                            636  
           ios      6.1.0                                              0  
                    6.1.1                                            154  
                    6.3.0                                              0  
2026-10-19 android  6.1.0                                              3  
                    6.2.0                                            668  
           ios      6.1.0                                              0  
                    6.1.1                                             54  
                    6.3.0                                              0

,reported_drives,eligible_drives
week,,
2026-10-12,21711,21711
2026-10-19,21595,21595


## 4. Mix and arrival diagnostics
Duration strata and device groups expose obvious differences in population mix; they do not remove confounding. Receipt timing relative to the following Monday in Copenhagen is shown as reporting availability, not event-to-ingestion lag. The first receipt is the earliest *observed* receipt. Reports can still arrive after the current cutoff.

In [5]:
cohort["duration_band"] = pd.cut(cohort["duration_s"], bins=[0,600,1800,float("inf")], labels=["under_10min", "10_to_30min", "over_30min"], right=False)
display(summarize(cohort, ["week", "platform", "duration_band"]).dropna(how="all"))
display(summarize(cohort, ["week", "platform", "device"]))
following_monday = (cohort["week"]+pd.Timedelta(days=7)).dt.tz_localize("Europe/Copenhagen")
cohort["received_after_following_monday"] = cohort["first_receipt_utc"].ge(following_monday)
display(cohort.groupby("week").agg(drives=("session_id","size"), received_after_following_monday=("received_after_following_monday","sum")))
# Sensitivity only: shifting all start clocks by +/-2 hours can move boundary sessions.
checks=[]
for shift in [-2,0,2]:
    shifted=(d["started_at"]+pd.Timedelta(hours=shift)).dt.to_period("W-SUN").dt.start_time
    for week in [previous_week,last_week]:
        sample=d.loc[d["eligible"] & shifted.eq(week)]
        checks.append({"uniform_clock_shift_hours":shift,"week":week,"drives":len(sample),"multiple_connection_pct":round(100*sample["multiple_connections"].mean(),3)})
display(pd.DataFrame(checks))
print("Uniform clock shifts are sensitivity checks, not validated timezone conversions.")

Uniform clock shifts are sensitivity checks, not validated timezone conversions.


eligible_drives  drivers  drives_with_multiple_connections  median_duration_s  p95_connections  multiple_connection_pct  \
week       platform duration_band                                                                                                                            
2026-10-12 android  under_10min               2477      867                               375              421.0              3.2                   15.139   
                    10_to_30min               5504     1071                               684             1031.0              3.0                   12.427   
                    over_30min                2112      839                               228             2446.0              3.0                   10.795   
           ios      under_10min               2764     1035                                64              421.0              1.0                    2.315   
                    10_to_30min               6483     1284                               116             1034.0              1.0                    1.789   
                    over_30min                2371     1005                                52             2514.0              1.0                    2.193   
2026-10-19 android  under_10min               2509      880                               372              419.0              4.0                   14.827   
                    10_to_30min               5636     1077                               717             1037.0              3.0                   12.722   
                    over_30min                1926      825                               231             2422.0              3.0                   11.994   
           ios      under_10min               2752     1041                                25              418.0              1.0                    0.908   
                    10_to_30min               6325     1286                                59             1016.0              1.0                    0.933   
                    over_30min                2447     1010                                18             2484.0              1.0                    0.736   

                                   drivers_with_multiple_connection_drives  
week       platform duration_band                                           
2026-10-12 android  under_10min                                        288  
                    10_to_30min                                        457  
                    over_30min                                         199  
           ios      under_10min                                         56  
                    10_to_30min                                         92  
                    over_30min                                          47  
2026-10-19 android  under_10min                                        277  
                    10_to_30min                                        479  
                    over_30min                                         202  
           ios      under_10min                                         20  
                    10_to_30min                                         41  
                    over_30min                                          16

eligible_drives  drivers  drives_with_multiple_connections  median_duration_s  p95_connections  multiple_connection_pct  \
week       platform device                                                                                                                             
2026-10-12 android  CD1                 3044      353                               398              993.0              3.0                   13.075   
                    CD2                 4332      488                               571              976.0              3.0                   13.181   
                    CD3                 1653      170                               201             1036.0              3.0                   12.160   
                    none                 971      114                               117             1030.0              3.0                   12.049   
                    unknown               93        1                                 0              827.0              1.0                    0.000   
           ios      CD1                 3564      407                                72             1005.0              1.0                    2.020   
                    CD2                 5016      595                               119              997.0              1.0                    2.372   
                    CD3                 1790      209                                29             1009.0              1.0                    1.620   
                    none                1141      125                                12             1038.0              1.0                    1.052   
                    unknown              107        1                                 0              854.0              1.0                    0.000   
2026-10-19 android  CD1                 3098      352                               407              951.0              3.0                   13.138   
                    CD2                 4277      484                               569              998.0              3.0                   13.304   
                    CD3                 1619      165                               213              989.0              3.0                   13.156   
                    none                1010      122                               131             1015.0              3.0                   12.970   
                    unknown               67        1                                 0              927.0              1.0                    0.000   
           ios      CD1                 3513      406                                38             1015.0              1.0                    1.082   
                    CD2                 5056      606                                46              973.0              1.0                    0.910   
                    CD3                 1762      209                                 9              998.0              1.0                    0.511   
                    none                1090      127                                 9             1001.0              1.0                    0.826   
                    unknown              103        1                                 0             1056.0              1.0                    0.000   

                             drivers_with_multiple_connection_drives  
week       platform device                                            
2026-10-12 android  CD1                                          211  
                    CD2                                          288  
                    CD3                                           93  
                    none                                          58  
                    unknown                                        0  
           ios      CD1                                           49  
                    CD2                                           78  
                    CD3                      

,drives,received_after_following_monday
week,,
2026-10-12,21711,125
2026-10-19,21595,77


,uniform_clock_shift_hours,week,drives,multiple_connection_pct
0,-2,2026-10-12,21711,6.996
1,-2,2026-10-19,21595,6.585
2,0,2026-10-12,21711,6.996
3,0,2026-10-19,21595,6.585
4,2,2026-10-12,21697,7.001
5,2,2026-10-19,21613,6.584


## 5. Review outcome
These checks support a provisional weekly connection-behavior comparison. Confirm timestamp semantics and what the count represents before promoting this to a release-health KPI. No causality or drive-failure claim is made. No dbt models or dashboards are created.

In [6]:
# Export aggregate findings only; no user/session identifiers.
def markdown_table(frame):
    table = frame.reset_index().copy()
    for col in table.columns:
        if pd.api.types.is_datetime64_any_dtype(table[col]):
            table[col] = table[col].dt.strftime("%Y-%m-%d")
    columns = [str(c) for c in table.columns]
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"]*len(columns)) + " |"]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in table.itertuples(index=False, name=None)]
    return "\n".join(lines)

change = weekly.loc[last_week,"multiple_connection_pct"]-weekly.loc[previous_week,"multiple_connection_pct"]
report = f"""# Provisional weekly connection behavior

Receipt cutoff: {as_of.isoformat()}. Offline analysis; no source refresh.
Weeks use provisional device wall-calendar start dates. This is a comparison of
recorded connection behavior, not a confirmed assessment of failed drives.

## Data validation

{markdown_table(audit.set_index('dataset'))}

Repeated drive reports agree on every observed business field; collapse receipt-only
replays to one session, retaining earliest/latest observed receipt and report count.
Warnings match canonical drives at {100*is_match.mean():.3f}%; {int((~is_match).sum())}
remain unmatched, with no user-ID mismatches among comparable matches.
There are {int(clock_future.sum())} future warning-clock flags and
{int(outside.sum())} matched warnings outside the drive interval.

## Last two complete calendar weeks

{markdown_table(weekly)}

The multiple-connection share changed by {change:.3f} percentage points.
Distinct-driver counts measure drivers with this recorded behavior, not drivers
proven to have had a failed drive. Driver totals are not additive across versions.

## Platform and release context

{markdown_table(platforms)}

{markdown_table(releases)}

The observed iOS 6.3.0 cohorts have no multiple-connection drives in these weeks.
That warrants checking counter semantics and instrumentation; it does not prove
zero interruptions. Adoption changes can alter aggregate rates even without
within-version improvement. Release comparisons are observational and confounded
by platform, duration, device, rollout population, and instrumentation.

## Timing and limitations

Most device end clocks are about two hours ahead of UTC receipt clocks in continental
countries and one hour ahead in GB, consistent with local clocks plus arrival delay.
This is evidence, not a timezone contract; country alone is insufficient. Do not
interpret negative clock differences as negative latency. See notebook duration/device
strata and +/-2-hour boundary sensitivity checks.

The saved cutoff includes reports arriving after their calendar week closed.
Further late reports can revise the results. Full pagination is not proof that all
attempted drives were reported, and the incremental refresh cannot guarantee older
backdated receipts. Confirm source timestamp and connection-count meanings before
using the candidate metric as a product-health KPI. Warning frequency and missing
actions do not establish missed alerts. No causal or monetary claim is supported.

Metric contract: [metric_definitions.md](metric_definitions.md).
"""
Path("reports").mkdir(exist_ok=True)
Path("reports/weekly_findings.md").write_text(report)
print("Saved reports/weekly_findings.md (aggregate tables only).")


Saved reports/weekly_findings.md (aggregate tables only).
